# Длинная линия в замедленной съёмке

Телеграфные уравнения решаются численно (метод прямых + Рунге–Кутта 4-го порядка), результат — мультфильм: **напряжение, ток, две бегущие волны и мощность** в каждой точке линии.

Линия по умолчанию — **пять длин волн** ($l = 5\lambda \approx 143$ м на 7 МГц). Источник — произвольная функция $e(t)$, нагрузка $R_l$ — любая.

**Как запускать:** `Kernel → Restart & Run All`. Расчёт — секунда, сборка анимации — ~30 с (она встраивается в ноутбук как HTML, поэтому файл весит десяток мегабайт).


## Что решается

$$-\frac{\partial U}{\partial x} = L' \frac{\partial I}{\partial t} + R' I, \qquad -\frac{\partial I}{\partial x} = C' \frac{\partial U}{\partial t} + G' U$$

Параметры линии задаются волновым сопротивлением и скоростью: $C' = \frac{1}{Z_0 v}$, $L' = \frac{Z_0}{v}$ (при 50 Ом и $2\cdot10^8$ м/с это 100 пФ/м и 250 нГн/м). Затухание задаётся одним числом $\beta$ по условию Хевисайда: $R' = \beta Z_0$, $G' = \beta/Z_0$ — тогда линия без дисперсии, амплитуда просто падает как $e^{-\beta x}$.

Граничные условия — как в лабораторной: на входе $U(0,t) = e(t) - R_s I(0,t)$, на выходе $U(l,t) = R_l I(l,t)$.

Схема: напряжение на узлах, ток в серединах ячеек (шахматка, как в FDTD), производные по $x$ — центральные разности, по $t$ — Рунге–Кутта 4. Устойчивость требует $dt < dx/v$; берётся $dt = 0{,}4\,dx/v$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams.update({
    "figure.dpi": 90,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
})
print("numpy:", np.__version__)


In [ ]:
# ============================== ПАРАМЕТРЫ ==============================
# --- линия ---
L_waves = 5.0         # длина линии в длинах волн (4…6 — самое интересное)
L_line  = None        # или задать метры числом — тогда L_waves не используется
Z0      = 50.0        # волновое сопротивление, Ом
v_wave  = 2.0e8       # скорость волны, м/с (≈ 2/3 c)
beta    = 0.0         # затухание, 1/м: 0 — без потерь; 0.01 → e^-1.4 на 143 м

# --- источник ---
f_src    = 7.0e6      # частота по умолчанию, Гц
amp      = 1.0        # амплитуда, В
Rs       = 50.0       # внутреннее сопротивление, Ом (50 = согласован с линией)

# --- нагрузка ---
Rl = np.inf           # холостой ход: полное отражение, ρ = +1
C_load = 0.0          # ёмкость параллельно нагрузке, Ф (0 — нет)
# ---- другие варианты нагрузки (раскомментируйте нужный) ----
# Rl = 50.0           # согласовано с линией, ρ = 0 — отражения нет
# Rl = 0.0            # короткое замыкание, ρ = −1
# Rl = 50.0 / 3       # ρ = −1/2
# Rl = 150.0          # ρ = +1/2  (3·Z0)
# Rl = 1e4            # почти холостой ход, ρ ≈ +0.99
# C_load = 100e-12    # 100 пФ: отражение растягивается на Z0·C = 5 нс
# C_load = 470e-12    # 470 пФ: Z0·C = 23.5 нс — сравнимо с пролётом
# C_load = 2e-12      # 2 пФ: столько у настоящего разъёма

# --- наблюдение ---
T_us       = 5.0      # сколько наблюдаем, мкс
SEC_PER_US = 4.0      # 1 мкс = столько секунд видео (4 → всего 20 с)

# --- сетка ---
N      = 800          # ячеек по длине (≈ 160 на длину волны)
n_rec  = 1200         # кадров записать
n_anim = 360          # кадров в анимации (больше кадров — плавнее)
# =======================================================================


In [ ]:
lam = v_wave / f_src
if L_line is None:
    L_line = L_waves * lam

tau    = L_line / v_wave
Cp     = 1.0 / (Z0 * v_wave)
Lp     = Z0 / v_wave
Rp     = beta * Z0
Gp     = beta / Z0
dx     = L_line / N
dt     = 0.4 * dx / v_wave

rho_l = 1.0 if not np.isfinite(Rl) else (-1.0 if Rl == 0 else (Rl - Z0) / (Rl + Z0))
rho_s = (Rs - Z0) / (Rs + Z0) if Rs > 0 else -1.0

print(f"λ = {lam:.2f} м;  линия l = {L_line:.1f} м = {L_line/lam:.2f} λ;  τ = {tau*1e9:.0f} нс")
print(f"нагрузка R_l = {Rl} Ом  →  ρ_l = {rho_l:+.2f};  источник R_s = {Rs} Ом  →  ρ_s = {rho_s:+.2f}")
print(f"сетка: N = {N}, dx = {dx*1000:.0f} мм, dt = {dt*1e12:.0f} пс, шагов {int(round(T_us*1e-6/dt))}")
if C_load > 0:
    print(f"ёмкость нагрузки {C_load*1e12:.0f} пФ → Z0·C = {Z0*C_load*1e9:.1f} нс"
          f" (для сравнения τ = {tau*1e9:.0f} нс)")


## Источник

Источник — **функция одного аргумента**: `e_source(t)` получает время в секундах, возвращает напряжение в вольтах. Вызывается на каждом шаге интегрирования, поэтому должна принимать и возвращать **числа**, не массивы.

В ячейке ниже — готовые заготовки (синус, ступенька, импульс, меандр, два способа подать дискретный сигнал) и пример своей функции. Выберите нужную строкой `src_fn = ...`.


In [ ]:
def src_sin(f=None, A=1.0, phase=0.0):
    f = f_src if f is None else f
    return lambda t: A * np.sin(2 * np.pi * f * t + phase)

def src_step(A=1.0, t0=0.0, rise=20e-9):
    # ступенька с конечным фронтом (идеальный разрыв в дискретной схеме «звенит»)
    return lambda t: A * 0.5 * (1.0 + np.tanh((t - t0) / rise))

def src_pulse(t0=0.0, width=40e-9, A=1.0):
    # гауссов горб: центр t0, полуширина width
    return lambda t: A * np.exp(-((t - t0) / width) ** 2)

def src_meander(f=None, A=1.0, duty=0.5, rise=20e-9):
    # меандр с конечными фронтами (идеальные фронты схема «размазывает» и звенит)
    f = f_src if f is None else f
    T = 1.0 / f

    t0 = 4.0 * rise          # сдвиг, чтобы в t = 0 источник был нулевым

    def g(t):
        n0 = int(np.floor(t / T))
        s = 0.0
        for n in range(max(0, n0 - 1), n0 + 1):
            s += 0.5 * (1.0 + np.tanh((t - n * T - t0) / rise))
            s -= 0.5 * (1.0 + np.tanh((t - n * T - t0 - duty * T) / rise))
        return A * s
    return g

def src_zoh(ts, us):
    # дискретный сигнал: отсчёты держатся до следующего (ЦАП без фильтра)
    ts = np.asarray(ts, float); us = np.asarray(us, float)
    def f(t):
        if t < ts[0]:
            return 0.0
        i = int(np.searchsorted(ts, t, side="right")) - 1
        return float(us[min(i, len(us) - 1)])
    return f

def src_linear(ts, us):
    # дискретный сигнал с линейной интерполяцией между отсчётами
    ts = np.asarray(ts, float); us = np.asarray(us, float)
    return lambda t: float(np.interp(t, ts, us))


# ==================== ВАШ ИСТОЧНИК ====================
def e_source(t):
    # t — секунды; вернуть напряжение в вольтах (одно число)
    return amp * np.sin(2 * np.pi * f_src * t)

src_fn = e_source

# ---- другие варианты (раскомментируйте нужный) ----
# src_fn = src_step(A=1.0, t0=0.1e-6)                    # ступенька
# src_fn = src_pulse(t0=0.2e-6, width=40e-9)             # одиночный горб
# src_fn = src_meander(f=7e6, duty=0.3)                  # меандр (уровни, не синус)
# src_fn = src_sin(f=3.5e6, A=2.0)                       # другая частота и амплитуда
# ts = np.arange(0, 2e-6, 25e-9)
# us = np.sign(np.sin(2 * np.pi * 5e6 * ts))
# src_fn = src_zoh(ts, us)                               # «цифровой» сигнал ±1 В

# ---- или совсем своя ----
# def e_source(t):
#     return 1.0 if (t / 1e-9) % 40 < 20 else 0.0        # меандр 25 МГц
# src_fn = e_source

print("e(0) =", src_fn(0.0), "| e(1 нс) =", src_fn(1e-9))


In [ ]:
def simulate(L_line=142.9, Z0=50.0, v=2e8, beta=0.0, Rs=50.0, Rl=np.inf,
             C_load=0.0, src=None, T=10e-6, N=800, n_rec=1200, cfl=0.4):
    """Телеграфные уравнения: шахматная сетка + Рунге–Кутта 4.

    U — на узлах x_i = i*dx (N+1 точка), I — в серединах ячеек (N точек).
    src — функция e(t). Возвращает t, x, xh, U[t, x], I[t, xh].
    """
    Cp = 1.0 / (Z0 * v)
    Lp = Z0 / v
    Rp = beta * Z0
    Gp = beta / Z0
    dx = L_line / N
    dt = cfl * dx / v
    n_steps = max(1, int(round(T / dt)))

    x = np.linspace(0.0, L_line, N + 1)
    xh = x[:-1] + dx / 2
    U = np.zeros(N + 1)
    I = np.zeros(N)

    src_fn = src if src is not None else (lambda t: 0.0)

    # Граничный узел задаём алгебраически (точно), если явной схеме хватает
    # запаса устойчивости; иначе — через ёмкость полуячейки.
    src_alg = (Rs <= 0) or (Rs * dt < 0.8 * Lp * dx)
    open_end = not np.isfinite(Rl)

    # Ёмкость нагрузки складывается с ёмкостью граничного узла. Пока она есть,
    # напряжение навязать нельзя — узел обязан интегрироваться.
    Cnode = Cp * dx / 2 + C_load
    invCnode = 1.0 / Cnode
    load_alg = (C_load == 0.0) and (not open_end) and (
        (Rl == 0) or (Rl * dt < 0.8 * Lp * dx))

    invCp = 1.0 / (Cp * dx)
    invLp = 1.0 / (Lp * dx)
    gp = Gp / Cp
    rp = Rp / Lp
    two = 2.0 * invCp

    def rhs(U, I, t):
        e = src_fn(t)
        dU = np.empty_like(U)
        dI = np.empty_like(I)
        dI[:] = -(U[1:] - U[:-1]) * invLp - rp * I
        dU[1:-1] = -(I[1:] - I[:-1]) * invCp - gp * U[1:-1]
        if src_alg:
            dU[0] = 0.0
        else:
            dU[0] = ((e - U[0]) / Rs - I[0]) * two - gp * U[0]
        if load_alg:
            dU[-1] = 0.0
        else:
            I_load = 0.0 if open_end else U[-1] / Rl
            dU[-1] = ((I[-1] - I_load) - Gp * dx * 0.5 * U[-1]) * invCnode
        return dU, dI

    rec = np.linspace(0, n_steps, n_rec).astype(int)
    rec_set = set(rec.tolist())
    Urec = np.empty((n_rec, N + 1))
    Irec = np.empty((n_rec, N))
    trec = np.empty(n_rec)

    t = 0.0
    k = 0
    dt2 = 0.5 * dt
    for step in range(n_steps + 1):
        if step in rec_set and k < n_rec:
            Urec[k] = U
            if src_alg:
                Urec[k, 0] = src_fn(t) - Rs * I[0]
            if load_alg:
                Urec[k, -1] = Rl * I[-1]
            Irec[k] = I
            trec[k] = t
            k += 1
        if step == n_steps:
            break
        aU, aI = rhs(U, I, t)
        bU, bI = rhs(U + dt2 * aU, I + dt2 * aI, t + dt2)
        cU, cI = rhs(U + dt2 * bU, I + dt2 * bI, t + dt2)
        dU, dI = rhs(U + dt * cU, I + dt * cI, t + dt)
        U = U + (dt / 6.0) * (aU + 2.0 * bU + 2.0 * cU + dU)
        I = I + (dt / 6.0) * (aI + 2.0 * bI + 2.0 * cI + dI)
        t += dt
        if src_alg:
            U[0] = (src_fn(t) - Rs * I[0]) if Rs > 0 else src_fn(t)
        if load_alg:
            U[-1] = Rl * I[-1]
    return trec[:k], x, xh, Urec[:k], Irec[:k]


In [ ]:
import time as _time
t0 = _time.time()
t, x, xh, U, I = simulate(L_line=L_line, Z0=Z0, v=v_wave, beta=beta, Rs=Rs, Rl=Rl,
                          C_load=C_load, src=src_fn, T=T_us * 1e-6, N=N, n_rec=n_rec)
print(f"готово за {_time.time() - t0:.1f} с | кадров {len(t)} × узлов {len(x)}")


## Анимация

Пять величин в каждом сечении линии:

| панель | что нарисовано | как читать |
|---|---|---|
| 1 | $U(x)$ и $Z_0 I(x)$ | то, что показали бы осциллограф и амперметр в каждом сечении |
| 2 | $A$ и $B$ — падающая и отражённая волны | **сердце лабораторной**: $A$ всегда бежит вправо, $B$ — влево. $A = \frac{U + Z_0 I}{2}$, $B = \frac{U - Z_0 I}{2}$ |
| 3 | $P = U I$ — мгновенная мощность | куда течёт энергия в данный момент |

Если $R_l = Z_0$, то $B$ строго ноль — вся волна уходит в нагрузку. Если $R_l = \infty$ или $0$, то $|B| = |A|$ и сумма двух встречных волн даёт стоячую: узел там, где они в противофазе.


In [ ]:
def make_animation(t, x, xh, U, I, Z0, T_us, sec_per_us, n_anim=360, dpi=68):
    idx = np.linspace(0, len(t) - 1, min(n_anim, len(t))).astype(int)

    # A, B и мощность считаем в серединах ячеек — там, где живёт ток
    Uh = 0.5 * (U[:, :-1] + U[:, 1:])
    A = 0.5 * (Uh + Z0 * I)
    B = 0.5 * (Uh - Z0 * I)
    P = Uh * I * 1000.0                      # мВт

    mUI = 1.15 * max(np.abs(U).max(), 1e-9)
    mAB = 1.15 * max(np.abs(A).max(), np.abs(B).max(), 1e-9)
    mP = 1.15 * max(np.abs(P).max(), 1e-9)

    fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(9.6, 7.4), sharex=True, dpi=dpi)
    l_u, = a1.plot([], [], color="#c0392b", lw=1.6, label="U(x)")
    l_i, = a1.plot([], [], color="#1f6fbf", lw=1.4, ls="--", label="Z₀·I(x)")
    l_a, = a2.plot([], [], color="#2e7d32", lw=1.6, label="A — падающая →")
    l_b, = a2.plot([], [], color="#8e24aa", lw=1.6, label="B — отражённая ←")
    l_p, = a3.plot([], [], color="#e07b00", lw=1.6)

    a1.set_ylim(-mUI, mUI); a1.set_ylabel("U, Z₀I, В")
    a2.set_ylim(-mAB, mAB); a2.set_ylabel("A, B, В")
    a3.set_ylim(-mP, mP);   a3.set_ylabel("P = U·I, мВт")
    a3.set_xlabel("x, м")
    for a in (a1, a2, a3):
        a.set_xlim(0.0, x[-1])
        a.axhline(0, color="k", lw=0.6)
        a.axvline(0, ls=":", color="0.5")
        a.axvline(x[-1], ls=":", color="0.5")
    a1.legend(loc="upper right", fontsize=8, ncol=2)
    a2.legend(loc="upper right", fontsize=8, ncol=2)
    a1.text(0.005, 0.86, "источник", transform=a1.transAxes, fontsize=8, color="0.45")
    a1.text(0.995, 0.86, "нагрузка", transform=a1.transAxes, fontsize=8,
            color="0.45", ha="right")
    stamp = a1.text(0.5, 0.86, "", transform=a1.transAxes, ha="center",
                    family="monospace", fontsize=11)

    def upd(i):
        k = idx[i]
        l_u.set_data(x, U[k])
        l_i.set_data(xh, Z0 * I[k])
        l_a.set_data(xh, A[k])
        l_b.set_data(xh, B[k])
        l_p.set_data(xh, P[k])
        stamp.set_text(f"t = {t[k]*1e9:6.0f} нс")
        return l_u, l_i, l_a, l_b, l_p, stamp

    fps = max(1.0, len(idx) / (T_us * sec_per_us))
    anim = FuncAnimation(fig, upd, frames=len(idx), interval=1000.0 / fps, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml(fps=fps))

make_animation(t, x, xh, U, I, Z0, T_us, SEC_PER_US, n_anim=n_anim)


## Что попробовать

| Что поменять | Что увидеть |
|---|---|
| `Rl = 50.0` | $B$ исчезает — чистая бегущая волна, $P > 0$ везде |
| `Rl = np.inf` (по умолчанию) | $B$ = $A$ по модулю: стоячая волна, десять узлов, мощность качается туда-обратно |
| `Rl = 0.0` | то же, но с переворотом знака: узел напряжения на конце |
| `Rl = 50/3` или `150` | частичное отражение: $|B| = |A|/2$ |
| `beta = 0.01` | амплитуда падает к концу как $e^{-\beta x}$ — видно по наклону $A$ |
| `C_load = 100e-12` | отражение перестаёт быть ступенькой — растягивается на Z₀C = 5 нс |
| `C_load = 470e-12` | Z₀C = 23.5 нс, сравнимо с пролётом — картина совсем другая |
| `src_fn = src_step(...)` | ступенька: видно, как фронт бежит и удваивается у холостого хода |
| `src_fn = src_pulse(...)` | горб: самый наглядный способ увидеть скорость $v$ и отражение |
| `f_src = 3.5e6` | вдвое меньше волн на той же линии |
| `L_waves = 3` | короче линия — меньше узлов |
| `T_us = 0.5`, `SEC_PER_US = 20` | крупный план: видно, как волна бежит |

**Про масштаб времени.** По умолчанию 1 мкс растягивается в 4 секунды, то есть все 5 мкс — в 20 секунд. Волна пробегает линию за $\tau \approx 0{,}7$ мкс, это почти 3 секунды видео — движение видно глазом. Период ($143$ нс) при таком масштабе занимает 0,57 с, так что колебание выглядит как 1–2 качания в секунду. Хотите ещё медленнее — увеличьте `SEC_PER_US` (и `n_anim`, чтобы не дёргалось).


## Ёмкость на нагрузке: сверка с главой 10

Резистор на конце отражает «ступенькой»: пришла волна — и сразу ушла назад в нужной пропорции. Ёмкость так не умеет: в первый миг она короткое замыкание, потом заряжается и становится холостым ходом. Отражение превращается из числа в **фильтр**.

В методичке (глава 10) для этого случая есть готовые формулы:

$$U(l,t) = \theta(t-\tau)\left(1 - e^{-(t-\tau)/Z_0C}\right)$$

$$U(0,t) = \tfrac{1}{2}\theta(t) + \tfrac{1}{2}\theta(t-2\tau)\left(1 - 2e^{-(t-2\tau)/Z_0C}\right)$$

Главное видно сразу: на выходе плавный подъём, а на входе — **провал** в момент $2\tau$, когда возвращается отражение от ещё не заряженного конденсатора. Проверим, что сеточное решение даёт то же самое.

Одна оговорка: формулы написаны для идеальной ступеньки. У нас фронт конечный (5 нс), поэтому провал чуть неполный — чем больше ёмкость, тем он ближе к нулю (на 5 нФ схема даёт уже $-0{,}006$ В).


In [ ]:
Cc = 2000e-12                                  # ёмкость нагрузки
Zc = Z0 * Cc                                   # постоянная времени, 5 нс
t0c = 0.05e-6                                  # ступенька включается в этот момент

src_c = src_step(A=1.0, t0=t0c, rise=5e-9)

tc, xc, xhc, Uc, Ic = simulate(L_line=L_line, Z0=Z0, v=v_wave, beta=0.0,
                               Rs=Rs, Rl=np.inf, C_load=Cc, src=src_c,
                               T=2.5e-6, N=N, n_rec=900)

ul = Uc[:, -1]                                 # напряжение на нагрузке
u0 = Uc[:, 0]                                  # напряжение на входе

# формулы главы 10, с поправкой на то, что источник включился в t0c
d_l = tc - t0c - tau
d_0 = tc - t0c - 2 * tau
f_l = np.where(d_l > 0, 1.0 - np.exp(-np.maximum(d_l, 0.0) / Zc), 0.0)
f_0 = 0.5 + np.where(d_0 > 0, 0.5 * (1.0 - 2.0 * np.exp(-np.maximum(d_0, 0.0) / Zc)), 0.0)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9.8, 5.2), sharex=True)
a1.plot(tc * 1e9, ul, color="#c0392b", lw=1.8, label="расчёт")
a1.plot(tc * 1e9, f_l, "k--", lw=1.0, label="формула главы 10")
a1.set_ylabel("U на нагрузке, В"); a1.set_ylim(-0.05, 1.15); a1.legend(fontsize=8)
a1.set_title(f"Ёмкость {Cc*1e12:.0f} пФ на конце: Z0·C = {Zc*1e9:.1f} нс, τ = {tau*1e9:.0f} нс")
a2.plot(tc * 1e9, u0, color="#1f6fbf", lw=1.8, label="расчёт")
a2.plot(tc * 1e9, f_0, "k--", lw=1.0, label="формула главы 10")
a2.set_ylabel("U на входе, В"); a2.set_xlabel("t, нс"); a2.legend(fontsize=8)
a2.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.show()

late_l = d_l > 3 * Zc
late_0 = d_0 > 3 * Zc
print(f"макс. расхождение с формулой: на нагрузке {np.abs(ul - f_l)[late_l].max():.3f} В, "
      f"на входе {np.abs(u0 - f_0)[late_0].max():.3f} В")
dip = (d_0 > 0) & (d_0 < 3 * Zc)
print(f"минимум на входе сразу после 2τ: {u0[dip].min():+.3f} В (у идеальной ступеньки было бы 0)")


## Проверка: расчёт против точного решения

Точное решение строится по характеристикам — ровно так, как в главе 3. Волна $A(0,t)$ рождается источником, добегает до нагрузки, отражается с коэффициентом $\rho_l$, возвращается, отражается от источника с $\rho_s$ — и так далее:

$$A(0,t) = \frac{1-\rho_s}{2}\,e(t) + \rho_s\rho_l\,e^{-2\beta l}\,A(0,\,t-2\tau)$$

Дальше всё восстанавливается: $A(x,t) = A(0,t-x/v)\,e^{-\beta x}$, $B(x,t) = B(0,t-(l-x)/v)\,e^{-\beta(l-x)}$, $U = A + B$.

Этот расчёт не зависит от того, какой у вас источник — синус, ступенька, импульс. У сигналов с очень крутыми фронтами расхождение больше: сетка размазывает разрыв на пару ячеек, поэтому смотрите на среднеквадратичное число, а не на максимальное. С ним и сравнивается сеточное решение.


In [ ]:
def exact_field(t, x, Z0, v, beta, Rs, Rl, src, m=8000):
    # точное решение по характеристикам; m — шагов на двойной пробег линии
    L = x[-1]
    tau = L / v
    rho_l = 1.0 if not np.isfinite(Rl) else (-1.0 if Rl == 0 else (Rl - Z0) / (Rl + Z0))
    rho_s = (Rs - Z0) / (Rs + Z0) if Rs > 0 else -1.0

    dt = 2.0 * tau / m
    n = int((t[-1] + L / v) / dt) + 2
    tt = np.arange(n) * dt
    ek = np.array([float(src(s)) for s in tt])

    A0 = np.zeros(n)
    gain = rho_s * rho_l * np.exp(-2.0 * beta * L)
    for k in range(n):
        A0[k] = 0.5 * (1.0 - rho_s) * ek[k] + (gain * A0[k - m] if k >= m else 0.0)
    B0 = np.zeros(n)
    B0[m:] = rho_l * np.exp(-2.0 * beta * L) * A0[:-m]

    Ue = np.empty((len(t), len(x)))
    for k in range(len(t)):
        tk = t[k]
        a = np.interp(tk - x / v, tt, A0) * np.exp(-beta * x)
        b = np.interp(tk + x / v, tt, B0) * np.exp(beta * x)
        Ue[k] = a + b
    return Ue


Ue = exact_field(t, x, Z0, v_wave, beta, Rs, Rl, src_fn)
# кадр t = 0 не сравниваем: в нём расчёт ещё не «увидел» источник
dU = np.abs(U[1:] - Ue[1:])
scale = max(np.abs(U[1:]).max(), 1e-9)
print(f"среднеквадратичное расхождение: {100 * np.sqrt((dU ** 2).mean()) / scale:.2f} %")
print(f"максимальное расхождение:       {100 * dU.max() / scale:.2f} %")


In [ ]:
k = int(0.7 * len(t))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.8, 3.6))
a1.plot(x, U[k], color="#c0392b", lw=1.6, label="расчёт")
a1.plot(x, Ue[k], "k--", lw=1.0, label="точное решение")
a1.set_xlabel("x, м"); a1.set_ylabel("U, В")
a1.set_title(f"момент t = {t[k] * 1e9:.0f} нс"); a1.legend(fontsize=8)
a2.semilogy(t[1:] * 1e6, np.maximum(dU.max(axis=1), 1e-9), color="#444")
a2.set_xlabel("t, мкс"); a2.set_ylabel("max|ΔU| по x, В")
a2.set_title("ошибка во времени")
plt.tight_layout()
plt.show()
